In [1]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 115 (delta 40), reused 98 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 5.05 MiB | 14.99 MiB/s, done.
Resolving deltas: 100% (40/40), done.


In [2]:
!ls

Dual_watermarking_Scheme  sample_data


In [3]:
%cd /content/Dual_watermarking_Scheme
!ls 

/content/Dual_watermarking_Scheme
data  notebooks  README.md  requirements.txt  setup.md	src


In [4]:
from huggingface_hub import login
login()

In [5]:
import torch
print(torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2.11.0+cpu
cpu


In [13]:
!ls src/utils

extract_data.py  key_manager.py  loadConfig.py	model.py  __pycache__


In [ ]:
from src.utils.loadConfig import load_config
config = load_config("secondLayer")

config

{'MODEL_NAME': 'facebook/opt-2.7b',
 'GREEN_FRACTION': 0.5,
 'PREV_TOKEN_SIZE': 5,
 'DETECTION_THRESHOLD': 0.6,
 'P_VALUE_THRESHOLD': 0.05}

In [10]:
config["MODEL_NAME"]

'facebook/opt-2.7b'

In [14]:
from src.utils.model import load_model

model,tokenizer,vocab_size = load_model(config["MODEL_NAME"])
model.to(device)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265


OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2560, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2560)
      (final_layer_norm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-31): 32 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
            (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
            (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
            (out_proj): Linear(in_features=2560, out_features=2560, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
          (final_layer_norm): Laye

In [16]:
from src.utils.key_manager import generate_key

key = generate_key()
print(f"secret key is generated: {key}")

secret key is generated: ac8e931d4f8545e6ce224e67743593b8b31f82cd44f35119c7f1d19c0118c011


In [17]:
from src.watermark.second_layer import PrivateWatermarkProcessor

processor = PrivateWatermarkProcessor(
    key=key,
    vocab_size=vocab_size,
    green_fraction=config["GREEN_FRACTION"],
    delta_private=0.7,
    prev_token_size=config["PREV_TOKEN_SIZE"]
)

print("watermark processor loaded .")

ImportError: cannot import name 'derive_set' from 'src.utils' (unknown location)

In [ ]:
from src.watermark.second_layer import generation_pipeline

prompt = "Artificial intelligence is changing the world because"

results = generation_pipeline(
    prompts=[prompt],
    model=model,
    tokenizer=tokenizer,
    processors=[processor],
    max_new_tokens=50,
    do_sample=True,
    temperature=1.0,
    top_p=0.9,
)

results

,id,prompt,plain_output,watermarked_output
0,0,Artificial intelligence is changing the world ...,Artificial intelligence is changing the world ...,Artificial intelligence is changing the world ...


In [ ]:
print("prompt = ")
print(results.iloc[0]["prompt"])

print("normal output is ",results.iloc[0]["plain_output"])
print("watermarked output is ",results.iloc[0]["watermarked_output"])

In [ ]:
from src.detection.kgw_detection import detect_private_watermark

watermarked_text = results.iloc[0]["watermarked_output"]

detection_result = detect_private_watermark(
    text=watermarked_text,
    tokenizer=tokenizer,
    key=key,
    vocab_size=vocab_size,
    green_fraction=config["GREEN_FRACTION"],
    prev_token_size=config["PREV_TOKEN_SIZE"],
    threshold=config["DETECTION_THRESHOLD"],
    p_value_threshold=config["P_VALUE_THRESHOLD"],
)

detection_result

In [ ]:
normal_text = results.iloc[0]["plain_output"]

normal_result = detect_private_watermark(
    text=normal_text,
    tokenizer=tokenizer,
    key=key,
    vocab_size=vocab_size,
    green_fraction=config["GREEN_FRACTION"],
    prev_token_size=config["PREV_TOKEN_SIZE"],
    threshold=config["DETECTION_THRESHOLD"],
    p_value_threshold=config["P_VALUE_THRESHOLD"],
)

print("Normal text:")
print("Watermark Detected:", normal_result["confirmed"])

print("Watermarked text:")
print("Watermark Detected:", detection_result["confirmed"])

In [ ]:
import gradio as gr
import pandas as pd

def generate_text(prompt, max_new_tokens=50):
    if not prompt or not prompt.strip():
        return "Please enter a prompt.", "Please enter a prompt."

    try:
        results = generation_pipeline(
            prompts=[prompt],
            model=model,
            tokenizer=tokenizer,
            processors=[processor],  # IMPORTANT: list
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=1.0,
            top_p=0.5,
        )

        return (
            results.iloc[0]["plain_output"],
            results.iloc[0]["watermarked_output"],
        )

    except Exception as e:
        print(f"Generation error: {e}")
        return (
            f"Generation error: {str(e)}",
            f"Generation error: {str(e)}",
        )

def detect_text(text):
    if not text or not text.strip():
        return 0.0, 0, 0, 1.0, False

    try:

        result = detect_private_watermark(
            text=text,
            tokenizer=tokenizer,
            key=key,
            vocab_size=vocab_size,
        )

        return (
            result["ownership_score"],
            result["match_count"],
            result["num_positions"],
            result["p_value"],
            result["confirmed"],
        )

    except Exception as e:

        print(f"Detection error: {e}")
        return (0.0,0,0,1.0,False)

with gr.Blocks(title="Private Watermark") as demo:

    gr.Markdown("# 🔐 Private Watermark Demo")
    gr.Markdown(
        "Generate normal and watermarked text, then test whether "
        "a text contains the private watermark."
    )

    with gr.Tab("Generation"):

        prompt = gr.Textbox(
            label="Prompt",
            placeholder="Artificial intelligence is changing the world because...",
            lines=4,
        )

        max_tokens = gr.Slider(
            minimum=10,
            maximum=200,
            value=50,
            step=1,
            label="Max New Tokens",
        )

        generate_button = gr.Button(
            "Generate",
            variant="primary",
        )

        plain_output = gr.Textbox(
            label="Normal Output",
            lines=8,
        )

        watermarked_output = gr.Textbox(
            label="Watermarked Output",
            lines=8,
        )

        generate_button.click(
            fn=generate_text,
            inputs=[prompt, max_tokens],
            outputs=[plain_output, watermarked_output],
        )

    with gr.Tab("Detection"):

        detection_input = gr.Textbox(
            label="Text to Analyze",
            placeholder="Paste text here...",
            lines=10,
        )

        detect_button = gr.Button(
            "Detect Watermark",
            variant="primary",
        )

        detection_result = gr.Textbox(
            label="Result",
            interactive=False,
        )

        detect_button.click(
            fn=detect_text,
            inputs=[detection_input],
            outputs=[detection_result],
        )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fdfef5989e4cdf164e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
